In [ ]:
#This file contains two codes: the second one follows the first and uses the output of the first.

In [ ]:
#Loads Sentinel-5P overpass timestamps for Tehran from a CSV file.
#Converts each overpass time to minutes since midnight to allow proper averaging of time values.
#Computes the mean overpass time across all available days.
#Converts the mean back to HH:MM format.
#Outputs a single representative local overpass time for Sentinel-5P over Tehran, useful for: Temporal consistency checks, Matching satellite data with ground observations, Daily or climatological NO₂ analysis timing

In [3]:
import pandas as pd
import os
import re
from datetime import datetime

# Paths
data_folder = r"C:\Users\intel\Downloads\New folder"
output_csv  = r"C:\Users\intel\Documents\SENTINEL-5P\tehran_overpass_times.csv"

# Tehran bounding box
lon_min, lon_max = 51.12, 51.61
lat_min, lat_max = 35.57, 35.83

# Helper: extract overpass time from filename (ROBUST)
def extract_overpass_time(filename):
    """
    Extract mean overpass datetime from Sentinel-5P filename
    using start and end granule times.
    """
    # find all patterns like 20240401T074712
    times = re.findall(r"\d{8}T\d{6}", filename)

    if len(times) < 2:
        raise ValueError(f"Cannot extract overpass time from filename: {filename}")

    dt_start = datetime.strptime(times[0], "%Y%m%dT%H%M%S")
    dt_end   = datetime.strptime(times[1], "%Y%m%dT%H%M%S")

    return dt_start + (dt_end - dt_start) / 2

# Main loop
records = []

for fname in os.listdir(data_folder):

    if not fname.lower().endswith(".csv"):
        continue

    file_path = os.path.join(data_folder, fname)
    df = pd.read_csv(file_path)

    # Check required columns
    if not {'Latitude', 'Longitude'}.issubset(df.columns):
        print(f"⚠ Skipped (missing lat/lon): {fname}")
        continue

    # Check if file covers Tehran
    inside_tehran = df[
        (df['Longitude'] >= lon_min) & (df['Longitude'] <= lon_max) &
        (df['Latitude']  >= lat_min) & (df['Latitude']  <= lat_max)
    ]

    if inside_tehran.empty:
        print(f"❌ Excluded (no Tehran coverage): {fname}")
        continue

    # Extract overpass time
    try:
        overpass_time = extract_overpass_time(fname)
    except ValueError as e:
        print(f"⚠ {e}")
        continue

    records.append({
        "file_name": fname,
        "overpass_time": overpass_time,
        "n_pixels_tehran": len(inside_tehran)
    })

    print(f"✅ Accepted: {fname} | pixels in Tehran: {len(inside_tehran)}")

# Save result
result_df = pd.DataFrame(records)
result_df.sort_values("overpass_time", inplace=True)

result_df.to_csv(output_csv, index=False)

print("\n🎉 Tehran overpass file created:")
print(output_csv)


❌ Excluded (no Tehran coverage): S5P_OFFL_L2__NO2____20231201T072645_20231201T090815_31779_03_020600_20231202T233004_Iran.csv
✅ Accepted: S5P_OFFL_L2__NO2____20231201T090815_20231201T104945_31780_03_020600_20231203T013142_Iran.csv | pixels in Tehran: 32
✅ Accepted: S5P_OFFL_L2__NO2____20231202T084909_20231202T103039_31794_03_020600_20231204T010309_Iran.csv | pixels in Tehran: 51
✅ Accepted: S5P_OFFL_L2__NO2____20231203T083003_20231203T101133_31808_03_020600_20231205T003721_Iran.csv | pixels in Tehran: 62
✅ Accepted: S5P_OFFL_L2__NO2____20231204T081057_20231204T095227_31822_03_020600_20231206T002201_Iran.csv | pixels in Tehran: 44
✅ Accepted: S5P_OFFL_L2__NO2____20231205T075151_20231205T093321_31836_03_020600_20231207T000312_Iran.csv | pixels in Tehran: 25
✅ Accepted: S5P_OFFL_L2__NO2____20231205T093321_20231205T111451_31837_03_020600_20231207T015610_Iran.csv | pixels in Tehran: 10
✅ Accepted: S5P_OFFL_L2__NO2____20231206T073246_20231206T091415_31850_03_020600_20231207T235033_Iran.csv |

In [ ]:
#Loads Sentinel-5P overpass timestamps for Tehran from a CSV file.
#Converts each overpass time to minutes since midnight to allow proper averaging of time values.
#Computes the mean overpass time across all available days.
#Converts the mean back to HH:MM format.
#Outputs a single representative local overpass time for Sentinel-5P over Tehran, useful for: Temporal consistency checks, Matching satellite data with ground observations, Daily or climatological NO₂ analysis timing

In [4]:
import pandas as pd

# Load overpass times
csv_path = r"C:\Users\intel\Documents\SENTINEL-5P\tehran_overpass_times.csv"
df = pd.read_csv(csv_path, parse_dates=["overpass_time"])

# Convert time to minutes since midnight
df["minutes"] = (
    df["overpass_time"].dt.hour * 60 +
    df["overpass_time"].dt.minute +
    df["overpass_time"].dt.second / 60
)

# Mean overpass time
mean_minutes = df["minutes"].mean()

mean_hour   = int(mean_minutes // 60)
mean_minute = int(mean_minutes % 60)

print("🛰 Mean Sentinel-5P overpass time over Tehran:")
print(f"{mean_hour:02d}:{mean_minute:02d}")


🛰 Mean Sentinel-5P overpass time over Tehran:
09:30
